# Textual Emotion Detection (`FEEL-IT`)
**Task:** Multi-label Emotion Classification (10 labels: Plutchik's 8 basic emotions + *Amore* + *Neutro*)  
**Base Encoder:** `MilaNLProc/feel-it-italian-emotion`  
**Datasets (`datasets/corpora/correctly_split/`):**
- `cleaned_emit_train_A.csv` (`EMit A`)
- `textual_only_cleaned.csv` (`MultiEMit` Textual-Only, `2,023` rows — split `80/20` into `me_train` = `1,618` rows and `me_test` = `405` rows)
- `multimodal_train_seed42.csv` (`MultiEMit` Multimodal Train Split, `516` rows — `0` overlap with `me_test` or `multimodal_test_seed42.csv`)

**Pipeline:**
1. **Step 1 (Model 1 — Without Correlations):**
   - Trained on `EMit A` (80% train split = `4,772` rows) + `MultiEMit` Textual train (`me_train` = `1,618` rows) = **`6,390` training rows** (raw text, no correlations).
   - Evaluated on **`me_test` (`405` rows, without correlations)**.
   - Saves checkpoint to `./checkpoints/emotion_textual_no_corr`.
2. **Step 2 (Model 2 — With Correlations):**
   - Initialized from **Step 1 (`./checkpoints/emotion_textual_no_corr`)**.
   - Trained on **`MultiEMit` Textual train (`me_train` = `1,618` rows) + `MultiEMit` Multimodal Train (`multimodal_train_seed42.csv` = `516` rows)** = **`2,134` correlated training rows** (`italian_correlations.py`).
   - Evaluated on the **exact same `MultiEMit` Textual test set (`me_test` = `405` rows) WITH correlations**.
   - Saves checkpoint to `./checkpoints/emotion_textual_corr`.


In [1]:
# Install CUDA-enabled PyTorch (supports GTX 1080 Ti / Pascal via CUDA 12.1) and all requirements
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
%pip install -q -r ../requirements.txt


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ['TQDM_NOTEBOOK'] = 'false'
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)
import tqdm.auto, tqdm.std
tqdm.auto.tqdm = tqdm.std.tqdm  # Force plain-text progress bars in VS Code

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


Using device: cuda


## 1. Configuration & Checkpoint Paths

In [3]:
BASE_MODEL_NAME = 'MilaNLProc/feel-it-italian-emotion'
FORCE_RETRAIN = True  # Set True to force retraining even if a checkpoint already exists

# Robust discovery of repository root containing datasets/
repo_root = Path.cwd().resolve()
for _ in range(5):
    if (repo_root / 'datasets').exists():
        break
    repo_root = repo_root.parent

DATASETS_DIR = repo_root / 'datasets' / 'corpora'
EMIT_A_PATH = str(DATASETS_DIR / 'emit' / 'cleaned_emit_train_A.csv')
MULTIEMIT_TEXTUAL_PATH = str(DATASETS_DIR / 'correctly_split' / 'textual_only_cleaned.csv')
MULTIMODAL_TRAIN_PATH = str(DATASETS_DIR / 'correctly_split' / 'multimodal_train_seed42.csv')

OUTPUT_DIR_NO_CORR = './checkpoints/emotion_textual_no_corr'
OUTPUT_DIR_CORR = './checkpoints/emotion_textual_corr'

EMOTION_LABELS = [
    'Amore', 'Anticipazione', 'Disgusto', 'Fiducia', 'Gioia',
    'Neutro', 'Paura', 'Rabbia', 'Sorpresa', 'Tristezza'
]
id2label = {idx: label for idx, label in enumerate(EMOTION_LABELS)}
label2id = {label: idx for idx, label in enumerate(EMOTION_LABELS)}
print('10 Emotion Labels:', EMOTION_LABELS)


10 Emotion Labels: ['Amore', 'Anticipazione', 'Disgusto', 'Fiducia', 'Gioia', 'Neutro', 'Paura', 'Rabbia', 'Sorpresa', 'Tristezza']


## 2. Tokenization & Multi-Label Evaluation Utilities

In [4]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

def preprocess_multilabel(examples):
    encoding = tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128,
    )
    labels_matrix = np.zeros((len(examples['text']), len(EMOTION_LABELS)), dtype=np.float32)
    for idx, label in enumerate(EMOTION_LABELS):
        labels_matrix[:, idx] = examples[label]
    encoding['labels'] = labels_matrix.tolist()
    return encoding

def predict_with_threshold(probs, threshold=0.4):
    preds = np.zeros(probs.shape, dtype=int)
    for i, row in enumerate(probs):
        above = np.where(row >= threshold)[0]
        if len(above) > 0:
            preds[i, above] = 1
        else:
            sorted_idx = np.argsort(row)[::-1]
            preds[i, sorted_idx[0]] = 1
            if row[sorted_idx[0]] - row[sorted_idx[1]] <= 0.1:
                preds[i, sorted_idx[1]] = 1
    return preds

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1.0 / (1.0 + np.exp(-logits))
    preds = predict_with_threshold(probs, threshold=0.4)
    return {
        'f1_micro': f1_score(labels, preds, average='micro', zero_division=0),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        'f1_weighted': f1_score(labels, preds, average='weighted', zero_division=0),
        'subset_accuracy': accuracy_score(labels, preds),
    }


## 3. Prepare Datasets (`EMit A`, `textual_only_cleaned.csv`, and `multimodal_train_seed42.csv`)

In [5]:
# 1. Load EMit A (raw text) and split 80/20
df_emit_a = pd.read_csv(EMIT_A_PATH)[['text'] + EMOTION_LABELS].dropna(subset=['text']).reset_index(drop=True)
for col in EMOTION_LABELS:
    df_emit_a[col] = df_emit_a[col].astype(int)

emit_a_train, emit_a_val = train_test_split(df_emit_a, test_size=0.2, random_state=SEED)
print(f'EMit A 80/20 Split -> Train: {len(emit_a_train)}, Val: {len(emit_a_val)}')

# 2. Load MultiEMit Textual-Only (textual_only_cleaned.csv) and Multimodal Train (multimodal_train_seed42.csv)
df_multiemit = pd.read_csv(MULTIEMIT_TEXTUAL_PATH)
df_mm_train = pd.read_csv(MULTIMODAL_TRAIN_PATH)
print(f'MultiEMit textual_only_cleaned rows: {len(df_multiemit)} | Multimodal Train rows: {len(df_mm_train)}')

def build_emotion_corr_text(df_in):
    """
    Exact correlation prefix generator from italian_correlations.py (target=True, irony=True, hate=True).
    """
    out_texts = []
    for _, row in df_in.iterrows():
        targets = []
        ironyhate = []

        if int(row.get('Contenuto', 0)) == 1:
            targets.append('contenuto')
        if int(row.get('Contenitore', 0)) == 1:
            targets.append('contenitore')

        if int(row.get('Ironia', 0)) == 1:
            ironyhate.append('ironia')
        if int(row.get('Sarcasmo', 0)) == 1:
            ironyhate.append('sarcasmo')
        if int(row.get('Offensivita', 0)) == 1:
            ironyhate.append('offensivita')

        text = str(row['text'])
        if targets:
            text = 'riguarda {} '.format(' e '.join(targets)) + text
        if ironyhate:
            text = 'contiene {} '.format(' e '.join(ironyhate)) + text

        text = 'il seguente testo ' + text
        out_texts.append(text)
    return out_texts

df_multiemit['text_corr'] = build_emotion_corr_text(df_multiemit)
df_mm_train['text_corr'] = build_emotion_corr_text(df_mm_train)

for col in EMOTION_LABELS:
    df_multiemit[col] = df_multiemit[col].astype(int)
    df_mm_train[col] = df_mm_train[col].astype(int)

# 80/20 split on textual_only_cleaned.csv (same test set used in both Step 1 and Step 2)
me_train, me_test = train_test_split(df_multiemit, test_size=0.2, random_state=SEED)
print(f'MultiEMit Textual 80/20 Split -> Train: {len(me_train)}, Test: {len(me_test)}')

# Step 1 Training Set: EMit A Train (4,772) + MultiEMit Textual Train (1,618) = 6,390 rows (NO correlations)
step1_train_no_corr = pd.concat(
    [emit_a_train[['text'] + EMOTION_LABELS], me_train[['text'] + EMOTION_LABELS]],
    ignore_index=True
).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

step1_test_no_corr = me_test[['text'] + EMOTION_LABELS].reset_index(drop=True)

# Step 2 Training Set: MultiEMit Textual Train (1,618) + Multimodal Train (516) = 2,134 rows (WITH correlations)
me_train_corr = me_train[['text_corr'] + EMOTION_LABELS].rename(columns={'text_corr': 'text'})
mm_train_corr = df_mm_train[['text_corr'] + EMOTION_LABELS].rename(columns={'text_corr': 'text'})

step2_train_corr = pd.concat(
    [me_train_corr, mm_train_corr],
    ignore_index=True
).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

# Step 2 Test Set: Same MultiEMit Textual Test Set (405 rows), WITH correlations
step2_test_corr = me_test[['text_corr'] + EMOTION_LABELS].rename(columns={'text_corr': 'text'}).reset_index(drop=True)

print(f'Step 1 Train (EMit A + MultiEMit Textual, No Corr): {len(step1_train_no_corr)} rows | Test: {len(step1_test_no_corr)} rows')
print(f'Step 2 Train (MultiEMit Textual + Multimodal Train, With Corr): {len(step2_train_corr)} rows | Test: {len(step2_test_corr)} rows')


EMit A 80/20 Split -> Train: 4772, Val: 1194
MultiEMit textual_only_cleaned rows: 2023 | Multimodal Train rows: 516
MultiEMit Textual 80/20 Split -> Train: 1618, Test: 405
Step 1 Train (EMit A + MultiEMit Textual, No Corr): 6390 rows | Test: 405 rows
Step 2 Train (MultiEMit Textual + Multimodal Train, With Corr): 2134 rows | Test: 405 rows


## 4. Step 1: Train `FEEL-IT` on `EMit A + MultiEMit Textual` (**Without Correlations**)

In [6]:
ds_step1 = DatasetDict({
    'train': Dataset.from_pandas(step1_train_no_corr),
    'test': Dataset.from_pandas(step1_test_no_corr),
})
encoded_step1 = ds_step1.map(
    preprocess_multilabel, batched=True, remove_columns=ds_step1['train'].column_names
)
encoded_step1.set_format('torch')

args_step1 = TrainingArguments(
    output_dir=OUTPUT_DIR_NO_CORR,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    save_total_limit=1,
    seed=SEED,
    report_to='none',
)

if os.path.exists(os.path.join(OUTPUT_DIR_NO_CORR, 'config.json')) and not FORCE_RETRAIN:
    print(f'Loading existing Step 1 checkpoint from {OUTPUT_DIR_NO_CORR} (skipping training)...')
    model_step1 = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR_NO_CORR)
    trainer_step1 = Trainer(
        model=model_step1,
        args=args_step1,
        eval_dataset=encoded_step1['test'],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )
else:
    model_step1 = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_NAME,
        problem_type='multi_label_classification',
        num_labels=len(EMOTION_LABELS),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )
    trainer_step1 = Trainer(
        model=model_step1,
        args=args_step1,
        train_dataset=encoded_step1['train'],
        eval_dataset=encoded_step1['test'],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )
    trainer_step1.train()
    trainer_step1.save_model(OUTPUT_DIR_NO_CORR)
    tokenizer.save_pretrained(OUTPUT_DIR_NO_CORR)
    print(f'Saved Step 1 (EMit A + MultiEMit Textual No-Corr) model to: {OUTPUT_DIR_NO_CORR}')

preds_step1 = trainer_step1.predict(encoded_step1['test'])
probs_step1 = 1.0 / (1.0 + np.exp(-preds_step1.predictions))
y_pred_no_corr = predict_with_threshold(probs_step1, threshold=0.4)
y_true_no_corr = preds_step1.label_ids.astype(int)

print('=== Step 1 Evaluation: MultiEMit Textual Test Set (WITHOUT Correlations) ===')
print(classification_report(y_true_no_corr, y_pred_no_corr, target_names=EMOTION_LABELS, digits=4, zero_division=0))


Map:   0%|          | 0/6390 [00:00<?, ? examples/s]

Map:   0%|          | 0/405 [00:00<?, ? examples/s]

d:\vsc_projects\MultiEMit\MultiEMit\.conda\Lib\site-packages\transformers\training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at MilaNLProc/feel-it-italian-emotion and are newly initialized because the shapes did not match:
- classifier.out_proj.weight: found shape torch.Size([4, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
- classifier.out_proj.bias: found shape torch.Size([4]) in the checkpoint and torch.Size([10]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\andre\AppData\Local\Temp\ipykernel_10280\2201710037.py:47: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,F1 Weighted,Subset Accuracy
1,No log,0.340700,0.341198,0.205999,0.260186,0.135802
2,0.363200,0.304775,0.463612,0.390866,0.436799,0.229630
3,0.271900,0.296236,0.480496,0.424080,0.460899,0.246914
4,0.223000,0.300710,0.508566,0.467520,0.495587,0.281481
5,0.188800,0.300020,0.514235,0.492814,0.511424,0.288889
6,0.188800,0.302044,0.509225,0.471055,0.502660,0.296296
7,0.161200,0.303206,0.520930,0.501111,0.518516,0.308642
8,0.143100,0.306188,0.529140,0.506598,0.527349,0.320988
9,0.130700,0.308196,0.540345,0.509167,0.537110,0.320988
10,0.124400,0.309461,0.531392,0.500381,0.528312,0.313580


Saved Step 1 (EMit A + MultiEMit Textual No-Corr) model to: ./checkpoints/concat_emotion_no_corr


=== Step 1 Evaluation: MultiEMit Textual Test Set (WITHOUT Correlations) ===
               precision    recall  f1-score   support

        Amore     0.6182    0.6939    0.6538        49
Anticipazione     0.4615    0.4800    0.4706        25
     Disgusto     0.5970    0.5405    0.5674        74
      Fiducia     0.6125    0.5904    0.6012        83
        Gioia     0.5041    0.6778    0.5782        90
       Neutro     0.4783    0.4231    0.4490        52
        Paura     0.3750    0.2500    0.3000        12
       Rabbia     0.4783    0.4074    0.4400        27
     Sorpresa     0.4659    0.4607    0.4633        89
    Tristezza     0.5435    0.5952    0.5682        42

    micro avg     0.5321    0.5488    0.5403       543
    macro avg     0.5134    0.5119    0.5092       543
 weighted avg     0.5318    0.5488    0.5371       543
  samples avg     0.5348    0.5247    0.5091       543



## 5. Step 2: Retrain Step 1 Model on `MultiEMit Textual Train + Multimodal Train` (**With Correlations**)

In [7]:
ds_step2 = DatasetDict({
    'train': Dataset.from_pandas(step2_train_corr),
    'test': Dataset.from_pandas(step2_test_corr),
})
encoded_step2 = ds_step2.map(
    preprocess_multilabel, batched=True, remove_columns=ds_step2['train'].column_names
)
encoded_step2.set_format('torch')

args_step2 = TrainingArguments(
    output_dir=OUTPUT_DIR_CORR,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    save_total_limit=1,
    seed=SEED,
    report_to='none',
)

if os.path.exists(os.path.join(OUTPUT_DIR_CORR, 'config.json')) and not FORCE_RETRAIN:
    print(f'Loading existing Step 2 checkpoint from {OUTPUT_DIR_CORR} (skipping training)...')
    model_step2 = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR_CORR)
    trainer_step2 = Trainer(
        model=model_step2,
        args=args_step2,
        eval_dataset=encoded_step2['test'],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )
else:
    print(f'Initializing Step 2 from Step 1 Concatenated checkpoint: {OUTPUT_DIR_NO_CORR}')
    model_step2 = AutoModelForSequenceClassification.from_pretrained(
        OUTPUT_DIR_NO_CORR,
        problem_type='multi_label_classification',
        num_labels=len(EMOTION_LABELS),
        id2label=id2label,
        label2id=label2id,
    )
    trainer_step2 = Trainer(
        model=model_step2,
        args=args_step2,
        train_dataset=encoded_step2['train'],
        eval_dataset=encoded_step2['test'],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )
    trainer_step2.train()
    trainer_step2.save_model(OUTPUT_DIR_CORR)
    tokenizer.save_pretrained(OUTPUT_DIR_CORR)
    print(f'Saved Step 2 (MultiEMit Textual + Multimodal Train With-Corr) model to: {OUTPUT_DIR_CORR}')

preds_step2 = trainer_step2.predict(encoded_step2['test'])
probs_step2 = 1.0 / (1.0 + np.exp(-preds_step2.predictions))
y_pred_corr = predict_with_threshold(probs_step2, threshold=0.4)
y_true_corr = preds_step2.label_ids.astype(int)

print('=== Step 2 Evaluation: Same MultiEMit Textual Test Set (WITH Correlations) ===')
print(classification_report(y_true_corr, y_pred_corr, target_names=EMOTION_LABELS, digits=4, zero_division=0))


Map:   0%|          | 0/2134 [00:00<?, ? examples/s]

Map:   0%|          | 0/405 [00:00<?, ? examples/s]

d:\vsc_projects\MultiEMit\MultiEMit\.conda\Lib\site-packages\transformers\training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Initializing Step 2 from Step 1 Concatenated checkpoint: ./checkpoints/concat_emotion_no_corr


C:\Users\andre\AppData\Local\Temp\ipykernel_10280\823033988.py:47: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_step2 = Trainer(


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,F1 Weighted,Subset Accuracy
1,No log,0.286181,0.542125,0.520608,0.535356,0.325926
2,No log,0.294629,0.522456,0.497541,0.518667,0.318519
3,No log,0.294632,0.549431,0.518106,0.545652,0.308642
4,0.167900,0.309032,0.548813,0.522692,0.545936,0.301235
5,0.167900,0.319048,0.539427,0.516697,0.538844,0.301235
6,0.167900,0.318170,0.556244,0.533193,0.551743,0.318519
7,0.167900,0.321762,0.552962,0.534874,0.550710,0.325926
8,0.100900,0.325560,0.552466,0.520314,0.548266,0.338272
9,0.100900,0.330412,0.548501,0.517436,0.545235,0.318519
10,0.100900,0.330603,0.543828,0.510736,0.540454,0.323457


Saved Step 2 (MultiEMit Textual + Multimodal Train With-Corr) model to: ./checkpoints/concat_emotion_corr


=== Step 2 Evaluation: Same MultiEMit Textual Test Set (WITH Correlations) ===
               precision    recall  f1-score   support

        Amore     0.5965    0.6939    0.6415        49
Anticipazione     0.5455    0.4800    0.5106        25
     Disgusto     0.5875    0.6351    0.6104        74
      Fiducia     0.5952    0.6024    0.5988        83
        Gioia     0.5670    0.6111    0.5882        90
       Neutro     0.4107    0.4423    0.4259        52
        Paura     0.6667    0.3333    0.4444        12
       Rabbia     0.5238    0.4074    0.4583        27
     Sorpresa     0.4624    0.4831    0.4725        89
    Tristezza     0.5273    0.6905    0.5979        42

    micro avg     0.5394    0.5672    0.5530       543
    macro avg     0.5483    0.5379    0.5349       543
 weighted avg     0.5406    0.5672    0.5507       543
  samples avg     0.5360    0.5377    0.5169       543



## 6. Summary Comparison Table (`MultiEMit` Textual 20% Test Split)

In [8]:
os.makedirs('./results', exist_ok=True)

rep_no_corr = classification_report(y_true_no_corr, y_pred_no_corr, target_names=EMOTION_LABELS, digits=4, zero_division=0)
rep_corr = classification_report(y_true_corr, y_pred_corr, target_names=EMOTION_LABELS, digits=4, zero_division=0)

summary_df = pd.DataFrame([
    {
        'Task': 'Emotion Detection (10-Label Multi-Label)',
        'Modality': 'Textual',
        'Notebook': 'train_textual_emotion.ipynb',
        'Configuration': 'FEEL-IT (EMit A + MultiEMit Textual) — No Correlations',
        'Micro F1': f1_score(y_true_no_corr, y_pred_no_corr, average='micro', zero_division=0),
        'Macro F1': f1_score(y_true_no_corr, y_pred_no_corr, average='macro', zero_division=0),
        'Weighted F1': f1_score(y_true_no_corr, y_pred_no_corr, average='weighted', zero_division=0),
        'Accuracy / Subset Acc': accuracy_score(y_true_no_corr, y_pred_no_corr),
    },
    {
        'Task': 'Emotion Detection (10-Label Multi-Label)',
        'Modality': 'Textual',
        'Notebook': 'train_textual_emotion.ipynb',
        'Configuration': 'FEEL-IT (Step 1 Model -> MultiEMit Textual + MM Train) With Correlations',
        'Micro F1': f1_score(y_true_corr, y_pred_corr, average='micro', zero_division=0),
        'Macro F1': f1_score(y_true_corr, y_pred_corr, average='macro', zero_division=0),
        'Weighted F1': f1_score(y_true_corr, y_pred_corr, average='weighted', zero_division=0),
        'Accuracy / Subset Acc': accuracy_score(y_true_corr, y_pred_corr),
    },
])

summary_df.to_csv('./results/train_textual_emotion_summary.csv', index=False)
with open('./results/train_textual_emotion_reports.txt', 'w', encoding='utf-8') as f_rep:
    f_rep.write('=== Step 1: (EMit A + MultiEMit Textual) WITHOUT Correlations ===\n' + rep_no_corr + '\n\n')
    f_rep.write('=== Step 2: (MultiEMit Textual + MM Train) WITH Correlations ===\n' + rep_corr + '\n')

display(summary_df.round(4))


,Task,Modality,Notebook,Configuration,Micro F1,Macro F1,Weighted F1,Accuracy / Subset Acc
0,Emotion Detection (10-Label Multi-Label),Textual (Concatenated),train_textual_emotion_concatenated.ipynb,Step 1: FEEL-IT on (EMit A + MultiEMit Textual...,0.5403,0.5092,0.5371,0.3210
1,Emotion Detection (10-Label Multi-Label),Textual (Concatenated -> Correlated Textual+MM...,train_textual_emotion_concatenated.ipynb,Step 2: Step 1 Model -> (MultiEMit Textual + M...,0.5530,0.5349,0.5507,0.3259
